# Autoresearch Experiment Analysis

Analysis of autonomous hyperparameter tuning results from `results.tsv`.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, val_bpb, memory_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["val_bpb"] = pd.to_numeric(df["valid_loss"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

Total experiments: 211
Columns: ['commit', 'valid_loss', 'memory_gb', 'status', 'description', 'val_bpb']


,commit,valid_loss,memory_gb,status,description,val_bpb
0,32a06d9,5.281082,23.9,KEEP,"baseline: fp32 eager, 4L d512 h32 dff1344 SwiG...",5.281082
1,5d08e50,5.123583,23.9,KEEP,E1 wall-clock LR schedule (warmup 3% of budget...,5.123583
2,73ace0a,4.828588,29.0,KEEP,E2 GPU-resident train data + on-device gather ...,4.828588
3,ea94336,4.017811,17.0,KEEP,E3 flash SDPA + int d_k fix + num_heads 32->8 ...,4.017811
4,268840b,4.000983,17.0,KEEP,E4 TF32 + fused AdamW + clip_grad_norm_(foreac...,4.000983
5,04238d7,3.925687,16.2,KEEP,E5 torch.compile(dynamic=False); 12290 steps (...,3.925687
6,70d671f,3.901143,16.0,KEEP,E6 weight tying (share emb<->output proj); par...,3.901143
7,2174fcf,3.893724,16.0,KEEP,E7 embedding init std 0.02 (was vocab-collapse...,3.893724
8,eb6c020,3.855055,26.5,KEEP,E8 batch_size 32->64 @ LR3e-3; loss -0.039 (mo...,3.855055
9,3e8bb4b,3.867072,47.4,DISCARD,E9 batch_size 64->128 @ LR3e-3; WORSE +0.012 (...,3.867072


In [3]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

Experiment outcomes:
status
DISCARD                                                                                                                            135
KEEP                                                                                                                                66
CRASH                                                                                                                                4
SKIP                                                                                                                                 1
NOISE-PROBE                                                                                                                          1
E127 RERUN OF E126 LINEAR DECAY: 3.553271 (VS 3.552029). CONFIRMS LINEAR BEATS COSINE ~0.005-0.006 (>2X NOISE). HEAD STAYS E126      1
E129 (3RD LINEAR RUN, SED NO-OP): 3.554170. LINEAR TRUE VALUE ~3.5532 (RUNS 3.552/3.553/3.554). HEAD STAYS E126                      1
E176 RERUN OF E167 (BEST): 

In [4]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    bpb = row["val_bpb"]
    desc = row["description"]
    print(f"  #{i:3d}  bpb={bpb:.6f}  mem={row['memory_gb']:.1f}GB  {desc}")

KEPT experiments (66 total):

  #  0  bpb=5.281082  mem=23.9GB  baseline: fp32 eager, 4L d512 h32 dff1344 SwiGLU RoPE, LR3e-3 warmup2000 cosT100k, bs32; 926 steps MFU0.62% (throughput-bound, warmup never completed; COLD page-cache)
  #  1  bpb=5.123583  mem=23.9GB  E1 wall-clock LR schedule (warmup 3% of budget, cosine->alpha_min at end); warmup now completes+anneals; 1501 steps (warm cache confound)
  #  2  bpb=4.828588  mem=29.0GB  E2 GPU-resident train data + on-device gather (math-identical); 2169 steps (+44% throughput vs E1)
  #  3  bpb=4.017811  mem=17.0GB  E3 flash SDPA + int d_k fix + num_heads 32->8 (d_k64) + bf16 autocast (fp32 CE/RoPE guards); 8761 steps (4x), VRAM 29->17GB
  #  4  bpb=4.000983  mem=17.0GB  E4 TF32 + fused AdamW + clip_grad_norm_(foreach); deleted ~60 lines custom code; 9460 steps (+8%)
  #  5  bpb=3.925687  mem=16.2GB  E5 torch.compile(dynamic=False); 12290 steps (+30%), MFU 8.24%
  #  6  bpb=3.901143  mem=16.0GB  E6 weight tying (share emb<->output proj);

## Val BPB Over Time

Track how the best (kept) val_bpb evolves as experiments progress. The running minimum shows the "frontier" -- the best result achieved so far.

In [5]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_bpb = valid.loc[0, "val_bpb"]

# Only plot points at or below baseline (the interesting region)
below = valid[valid["val_bpb"] <= baseline_bpb + 0.0005]

# Plot discarded as faint background dots
disc = below[below["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_bpb"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = below[below["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_bpb"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running minimum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_bpb = valid.loc[kept_mask, "val_bpb"]
running_min = kept_bpb.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, bpb in zip(kept_idx, kept_bpb):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (idx, bpb),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation BPB (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from just below best to just above baseline
best_bpb = kept_bpb.min()
margin = (baseline_bpb - best_bpb) * 0.15
ax.set_ylim(best_bpb - margin, baseline_bpb + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

Saved to progress.png


## Summary Statistics

In [6]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_bpb = df.iloc[0]["val_bpb"]
best_bpb = kept["val_bpb"].min()
best_row = kept.loc[kept["val_bpb"].idxmin()]

print(f"Baseline val_bpb:  {baseline_bpb:.6f}")
print(f"Best val_bpb:      {best_bpb:.6f}")
print(f"Total improvement: {baseline_bpb - best_bpb:.6f} ({(baseline_bpb - best_bpb) / baseline_bpb * 100:.2f}%)")
print(f"Best experiment:   {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: bpb={row['val_bpb']:.6f}  {desc}")

Baseline val_bpb:  5.281082
Best val_bpb:      3.466811
Total improvement: 1.814271 (34.35%)
Best experiment:   E185 AdamW beta1 0.6->0.5 @ batch160; -0.0002 (FLAT, trend flattening; beta1 optimum plateau ~0.5-0.6). new best 3.466811

Cumulative effort per improvement:
  Experiment #  0: bpb=5.281082  baseline: fp32 eager, 4L d512 h32 dff1344 SwiGLU RoPE, LR3e-3 warmup2000 cosT100k, bs32; 926 steps MFU0.62% (throughput-bound, warmup never completed; COLD page-cache)
  Experiment #  1: bpb=5.123583  E1 wall-clock LR schedule (warmup 3% of budget, cosine->alpha_min at end); warmup now completes+anneals; 1501 steps (warm cache confound)
  Experiment #  2: bpb=4.828588  E2 GPU-resident train data + on-device gather (math-identical); 2169 steps (+44% throughput vs E1)
  Experiment #  3: bpb=4.017811  E3 flash SDPA + int d_k fix + num_heads 32->8 (d_k64) + bf16 autocast (fp32 CE/RoPE guards); 8761 steps (4x), VRAM 29->17GB
  Experiment #  4: bpb=4.000983  E4 TF32 + fused AdamW + clip_grad_no

## Top Hits (Kept Experiments by Improvement)

In [7]:
# Each kept experiment's delta is measured vs the previous kept experiment's bpb
# (since experiments are cumulative -- each one builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_bpb"] = kept["val_bpb"].shift(1)
kept["delta"] = kept["prev_bpb"] - kept["val_bpb"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'BPB':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_bpb']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")

Rank     Delta         BPB  Description
--------------------------------------------------------------------------------
   1  +0.810777  4.017811  E3 flash SDPA + int d_k fix + num_heads 32->8 (d_k64) + bf16 autocast (fp32 CE/RoPE guards); 8761 steps (4x), VRAM 29->17GB
   2  +0.294995  4.828588  E2 GPU-resident train data + on-device gather (math-identical); 2169 steps (+44% throughput vs E1)
   3  +0.157499  5.123583  E1 wall-clock LR schedule (warmup 3% of budget, cosine->alpha_min at end); warmup now completes+anneals; 1501 steps (warm cache confound)
   4  +0.075296  3.925687  E5 torch.compile(dynamic=False); 12290 steps (+30%), MFU 8.24%
   5  +0.059383  3.742895  E25 MUON optimizer (Newton-Schulz orthogonalized momentum) for 2D hidden weights, AdamW for emb+norms, muon_lr 0.02; loss -0.059 despite 8% fewer tokens! biggest win since throughput. new best, beats non-compliant E13
   6  +0.056546  3.495483  E150 COMPILE training cross_entropy; HUGE WIN -0.057 (MFU 12->16.9, tokens 